# Manipulação de Dados: Junção, Combinação e Reestruturação

Muitas vezes, os dados que necessitamos estão espalhados em diversos ficheiros ou bases de dados, e precisam ser organizados de forma a poderem ser analisados de forma mais fácil.

Vamos começar por rever um pouco sobre índices, e falar de indexação hierárquica:

In [1]:
import pandas as pd
import numpy as np

pd.options.display.max_columns = 999

## Índices

Os índices são um objecto do pandas que até agora só "utilizamos", sem nunca termos falado que era, de facto, um objecto do pandas.

O índices guardam as etiquetas dos eixos, e podem-se definir através do construtor `Index`:

In [2]:
etiquetas = pd.Index([2020, 2021, 2022, 2023, 2024], name='anos')
etiquetas

Index([2020, 2021, 2022, 2023, 2024], dtype='int64', name='anos')

Posteriormente pode-se utilizar este objecto como índice de uma `Series` ou `DataFrame`:

In [3]:
vendas = [123, 232, 287, 310, 350]
s = pd.Series(vendas, index=etiquetas)
s

anos
2020    123
2021    232
2022    287
2023    310
2024    350
dtype: int64

Atenção que se definimos um índice com `n` etiquetas, quando utilizamos essa índice numa `Series` ou `DataFrame`, precisamos de ter `n` valores, caso contrário iremos provocar um exceção do tipo `ValueError`.

In [4]:
try:
    pd.Series([1,2,3], index=etiquetas)
except ValueError as e:
    print(f'ERROR: {e}')

ERROR: Length of values (3) does not match length of index (5)


In [5]:
lucro = [45, 67, 79, 90, 110]
valores = np.array([vendas, lucro]).transpose()     # Criar um array de 5x2
valores

array([[123,  45],
       [232,  67],
       [287,  79],
       [310,  90],
       [350, 110]])

In [6]:
df = pd.DataFrame(valores, index=etiquetas)
df

,0,1
anos,,
2020,123,45
2021,232,67
2022,287,79
2023,310,90
2024,350,110


A matriz `valores` acima está a ser definida através do `numpy` de forma a se conseguir fazer a transposta com a função `transpose` do `numpy`.

In [7]:
colunas = pd.Index(['Vendas', 'Lucro'], name='operações')
colunas

Index(['Vendas', 'Lucro'], dtype='str', name='operações')

In [8]:
pd.DataFrame(valores, index=etiquetas, columns=colunas)

operações,Vendas,Lucro
anos,,
2020,123,45
2021,232,67
2022,287,79
2023,310,90
2024,350,110


No exemplo acima podemos ver que o nomes dos índices aparecem no canto superior esquerdo do `DataFrame`, quer para as linhas (`anos`) quer para as colunas (`operações`), não confundir esses nomes com valores do `DataFrame`.

### Indexação Hierárquica

Até agora representamos dados uni-dimensionais e bidimensionais, mas por vezes é necessário representar estruturas de dimensões superiores, i.e., dados com mais do que dois índices. Isso pode ser feito através da indexação hierárquica, que permite criar índices de dois (ou mais) níveis em qualquer um dos eixos.

Podemos começar por passar uma lista de listas para o parâmetro `index`, quando se está criar uma `Series`:

In [9]:
nums = pd.Series(np.random.normal(size=9),
                 index=[
                     ['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'],
                     [1, 2, 3, 1, 3, 2, 3, 2, 1]
                 ])
nums

a  1    0.473717
   2    0.539191
   3    0.521046
b  1    2.038778
   3   -0.263508
c  2    0.109191
   3    0.570416
d  2    0.701510
   1   -0.034836
dtype: float64

Como se pode observar, existem "buracos" nos índices do primeiro nível. Isso é apenas uma forma mais "agradável" de apresentar  um índice hierárquico, e basicamente os "buracos" querem dizer que é o mesmo valor que aparece acima.

Mas a forma mais fácil de criar um índice hierárquico é utilizando os métodos `from_arrays`, `from_tuples` e `from_product`, do objecto `MultiIndex`:

Por exemplo, suponha que se pretende criar uma `Series` com a seguinte tabela, onde existem dois níveis de índices: o "Continente" e o "País":

![](imagens\multilinhas.png)

In [10]:
array = [
    ['Asia', 'Asia', 'Asia', 'Europa', 'Europa', 'Europa', 'América', 'América'],
    ['China', 'India', 'Japão', 'Portugal', 'Alemanha', 'Reino Unido', 'U.S.A.', 'Canadá']
]

pd.MultiIndex.from_arrays(array, names=['Continentes', 'Países'])

MultiIndex([(   'Asia',       'China'),
            (   'Asia',       'India'),
            (   'Asia',       'Japão'),
            ( 'Europa',    'Portugal'),
            ( 'Europa',    'Alemanha'),
            ( 'Europa', 'Reino Unido'),
            ('América',      'U.S.A.'),
            ('América',      'Canadá')],
           names=['Continentes', 'Países'])

In [11]:
tuplo = [('Asia','China'), ('Asia', 'India'), ('Asia','Japão'),
         ('Europa', 'Portugal'), ('Europa', 'Alemanha'), ('Europa', 'Reino Unido'),
         ('América', 'U.S.A.'), ('América', 'Canadá')
        ]
indice = pd.MultiIndex.from_tuples(tuplo, names=['Continentes', 'Países'])
indice

MultiIndex([(   'Asia',       'China'),
            (   'Asia',       'India'),
            (   'Asia',       'Japão'),
            ( 'Europa',    'Portugal'),
            ( 'Europa',    'Alemanha'),
            ( 'Europa', 'Reino Unido'),
            ('América',      'U.S.A.'),
            ('América',      'Canadá')],
           names=['Continentes', 'Países'])

E depois pode-se criar a `Series`:

In [12]:
pib = [1999, 916, 1374, 359, 1497, 1665, 1657, 1286]
paises = pd.Series(pib, index=indice, name='PIB')
paises

Continentes  Países     
Asia         China          1999
             India           916
             Japão          1374
Europa       Portugal        359
             Alemanha       1497
             Reino Unido    1665
América      U.S.A.         1657
             Canadá         1286
Name: PIB, dtype: int64

Num `DataFrame`, as linhas e colunas são perfeitamente simétricas, i.e., tal como as linhas podem ter vários níveis de índices, as colunas também podem ter. Vamos neste caso utilizar o método `from_product`, que criar os índices fazendo o "produto" dos elementos das listas:

In [13]:
linhas = pd.MultiIndex.from_product([[2023, 2022], [2, 1]])

colunas = pd.MultiIndex.from_product([['Roberto', 'Ana', 'José'],
                                      ['BPM', 'Temperatura']],
                                     names=['paciente', 'medida'])

Vamos criar uns dados fictícios:

In [14]:
data = np.abs(np.round(np.random.randn(4,6), 1))
data[:, ::2] *= 30
data += 37

In [15]:
saude = pd.DataFrame(data, index=linhas, columns=colunas)
saude

paciente Roberto               Ana              José            
medida       BPM Temperatura   BPM Temperatura   BPM Temperatura
2023 2      58.0        38.5  46.0        37.5  85.0        38.5
     1      67.0        37.5  67.0        37.7  58.0        37.8
2022 2      61.0        37.6  46.0        37.4  79.0        37.1
     1      43.0        37.6  61.0        38.1  70.0        37.7

Estes dados são basicamente uma estrutura de 4 dimensões (paciente, tipo de medida, ano e numero da consulta), representada numa estrutura bidimensional.

Já agora, como nas linhas não demos nomes aos índices, podemos fazê-lo através do atributo `index.names` (também existe o `columns.names` para as colunas):

In [16]:
saude.index.names = ['ano', 'consulta']
saude

paciente      Roberto               Ana              José            
medida            BPM Temperatura   BPM Temperatura   BPM Temperatura
ano  consulta                                                        
2023 2           58.0        38.5  46.0        37.5  85.0        38.5
     1           67.0        37.5  67.0        37.7  58.0        37.8
2022 2           61.0        37.6  46.0        37.4  79.0        37.1
     1           43.0        37.6  61.0        38.1  70.0        37.7

#### Aceder aos elementos

Tal como no caso de índices "simples", aceder aos elementos através de índices hierárquicos, é simples, e ajuda pensar nos índices como dimensões "extra".

Vamos começar com aceder às `Series`:

In [17]:
paises

Continentes  Países     
Asia         China          1999
             India           916
             Japão          1374
Europa       Portugal        359
             Alemanha       1497
             Reino Unido    1665
América      U.S.A.         1657
             Canadá         1286
Name: PIB, dtype: int64

Podemos aceder a um elemento indicando o índices de primeiro nível e de segundo nível:

In [18]:
paises['Europa', 'Portugal']

np.int64(359)

Mas o `MultiIndex` permite indexação parcial, ou seja, indexar indicando apenas um dos níveis. O resultado é outra `Series`, com os outros índices dos outros níveis mantidos:

In [19]:
paises['Europa']

Países
Portugal        359
Alemanha       1497
Reino Unido    1665
Name: PIB, dtype: int64

Se quiser indicar apenas um índice de um nível abaixo, deve-se utilizar `:` nos níveis acima, para indicar que são _todos_ os elementos desses níveis:

In [20]:
paises[:,'Portugal']

Continentes
Europa    359
Name: PIB, dtype: int64

No caso das `Series`, pode-se obter exatamente o mesmo resultado utilizando os atributos `loc` e `iloc`:

In [21]:
paises.loc['Europa']

Países
Portugal        359
Alemanha       1497
Reino Unido    1665
Name: PIB, dtype: int64

In [22]:
paises.loc[:, 'Portugal']

Continentes
Europa    359
Name: PIB, dtype: int64

No caso de um `DataFrame`, é importante lembrar que as colunas são a principal referência dos mesmos, logo utilizar os índices como fizemos nas `Series` vai referenciar as colunas:

In [23]:
saude

paciente      Roberto               Ana              José            
medida            BPM Temperatura   BPM Temperatura   BPM Temperatura
ano  consulta                                                        
2023 2           58.0        38.5  46.0        37.5  85.0        38.5
     1           67.0        37.5  67.0        37.7  58.0        37.8
2022 2           61.0        37.6  46.0        37.4  79.0        37.1
     1           43.0        37.6  61.0        38.1  70.0        37.7

In [24]:
saude['Roberto']

medida          BPM  Temperatura
ano  consulta                   
2023 2         58.0         38.5
     1         67.0         37.5
2022 2         61.0         37.6
     1         43.0         37.6

In [25]:
saude['Roberto', 'BPM']

ano   consulta
2023  2           58.0
      1           67.0
2022  2           61.0
      1           43.0
Name: (Roberto, BPM), dtype: float64

Para aceder aos índices das linhas devemos utilizar os atributos `loc` e `iloc`:

In [26]:
saude.loc[2023]

paciente Roberto               Ana              José            
medida       BPM Temperatura   BPM Temperatura   BPM Temperatura
consulta                                                        
2           58.0        38.5  46.0        37.5  85.0        38.5
1           67.0        37.5  67.0        37.7  58.0        37.8

In [27]:
saude.loc[2023, 2]

paciente  medida     
Roberto   BPM            58.0
          Temperatura    38.5
Ana       BPM            46.0
          Temperatura    37.5
José      BPM            85.0
          Temperatura    38.5
Name: (2023, 2), dtype: float64

Para indicar os vários níveis dos índices das linhas e das colunas, podem-se utilizar tuplos, ou um índice do tipo `IndexSlice`.

Utilizando tuplos para os índices das linhas e colunas:

In [28]:
saude.loc[(2022, 1), ('Ana', 'Temperatura')]

np.float64(38.1)

Também de pode utilizar os dois-pontos (`:`) para indicar que se pretende todos os elementos de uma das dimensões (linhas/colunas):

In [29]:
saude.loc[:, ('Ana', 'BPM')]

ano   consulta
2023  2           46.0
      1           67.0
2022  2           46.0
      1           61.0
Name: (Ana, BPM), dtype: float64

No entanto, não se pode utilizar diretamente o sinal `:` para indicar que são todos os elementos desse sub-nível, como se fez nas `Series`, pois vai provocar um erro de sintaxe:

In [30]:
#saude.loc[(2022,1), ('Ana',:)]

Mas podemos fazer isso criando um índice do tipo `IndexSlice`, e  utilizando esse índice para indicar os elementos que pretendemos:

In [31]:
idx = pd.IndexSlice

In [32]:
saude.loc[idx[2023,:], idx[:,'BPM']]

paciente      Roberto   Ana  José
medida            BPM   BPM   BPM
ano  consulta                    
2023 2           58.0  46.0  85.0
     1           67.0  67.0  58.0

In [33]:
saude.loc[idx[2023,1], idx['Roberto', 'BPM']]

np.float64(67.0)

### Trocar e reordenar os níveis

Caso os níveis hierárquicos não estejam ordenados, o acesso e seleção dos elementos no pandas não é tão eficiente como podia ser se os mesmos estivessem ordenados, por isso é importante, principalmente em grandes _datasets_, que os índices estejam ordenados lexicograficamente a partir do nível mais exterior.

Isso é feito utilizando o método `sort_index`, que, por defeito, ordena os índices das linhas, começando no mais exterior (`level=0`), mas caso se pretenda ordenar a partir de um nível mais abaixo, pode-se indicar esse nível no parâmetro `level` (o nível `0` é o primeiro nível dos índices).

In [34]:
saude

paciente      Roberto               Ana              José            
medida            BPM Temperatura   BPM Temperatura   BPM Temperatura
ano  consulta                                                        
2023 2           58.0        38.5  46.0        37.5  85.0        38.5
     1           67.0        37.5  67.0        37.7  58.0        37.8
2022 2           61.0        37.6  46.0        37.4  79.0        37.1
     1           43.0        37.6  61.0        38.1  70.0        37.7

In [35]:
saude.sort_index(axis=1)

paciente        Ana              José             Roberto            
medida          BPM Temperatura   BPM Temperatura     BPM Temperatura
ano  consulta                                                        
2023 2         46.0        37.5  85.0        38.5    58.0        38.5
     1         67.0        37.7  58.0        37.8    67.0        37.5
2022 2         46.0        37.4  79.0        37.1    61.0        37.6
     1         61.0        38.1  70.0        37.7    43.0        37.6

In [36]:
saude.sort_index(level=1)

paciente      Roberto               Ana              José            
medida            BPM Temperatura   BPM Temperatura   BPM Temperatura
ano  consulta                                                        
2022 1           43.0        37.6  61.0        38.1  70.0        37.7
2023 1           67.0        37.5  67.0        37.7  58.0        37.8
2022 2           61.0        37.6  46.0        37.4  79.0        37.1
2023 2           58.0        38.5  46.0        37.5  85.0        38.5

Claro que dessa forma (ordenar por um índice que não o mais exterior) pode-se estar a perder um pouco da performance do pandas, e para evitar isso podemos trocar a ordem dos índices através do método `swaplevel`, e dessa forma ordenar sempre pelo nível mais exterior.

In [37]:
saude.swaplevel('ano', 'consulta').sort_index()

paciente      Roberto               Ana              José            
medida            BPM Temperatura   BPM Temperatura   BPM Temperatura
consulta ano                                                         
1        2022    43.0        37.6  61.0        38.1  70.0        37.7
         2023    67.0        37.5  67.0        37.7  58.0        37.8
2        2022    61.0        37.6  46.0        37.4  79.0        37.1
         2023    58.0        38.5  46.0        37.5  85.0        38.5

In [38]:
saude.swaplevel('paciente', 'medida', axis=1)

medida            BPM Temperatura   BPM Temperatura   BPM Temperatura
paciente      Roberto     Roberto   Ana         Ana  José        José
ano  consulta                                                        
2023 2           58.0        38.5  46.0        37.5  85.0        38.5
     1           67.0        37.5  67.0        37.7  58.0        37.8
2022 2           61.0        37.6  46.0        37.4  79.0        37.1
     1           43.0        37.6  61.0        38.1  70.0        37.7

Por vezes pode ser necessário efetuar várias operações com os índices, como sejam trocar linhas/colunas (fazer a transposta), passar colunas de valores para índices ou vice-versa. 

Considere o seguinte `DataFrame`, onde existe um índice hierárquico nas linhas:

In [39]:
df1 = pd.DataFrame(paises)
df1

PIB
Continentes Países           
Asia        China        1999
            India         916
            Japão        1374
Europa      Portugal      359
            Alemanha     1497
            Reino Unido  1665
América     U.S.A.       1657
            Canadá       1286

Passar índices "para dentro" do `DataFrame`, na prática, transformando os índices em colunas do `DataFrame`, pode ser feito recorrendo ao método `reset_index`. Este método vai criar um novo índice sequencial no `DataFrame` e por defeito passa os índices existentes para colunas do `DataFrame`, e permite também dar nomes às "novas" colunas, através do parâmetro `names`:

In [40]:
df2 = df1.reset_index()
df2

,Continentes,Países,PIB
0,Asia,China,1999
1,Asia,India,916
2,Asia,Japão,1374
3,Europa,Portugal,359
4,Europa,Alemanha,1497
5,Europa,Reino Unido,1665
6,América,U.S.A.,1657
7,América,Canadá,1286


In [41]:
df1.reset_index(names=['nivel1', 'nivel2'])

,nivel1,nivel2,PIB
0,Asia,China,1999
1,Asia,India,916
2,Asia,Japão,1374
3,Europa,Portugal,359
4,Europa,Alemanha,1497
5,Europa,Reino Unido,1665
6,América,U.S.A.,1657
7,América,Canadá,1286


Outra opções deste método é remover por completo os índices anteriores, caso se utilize o parâmetro `drop=True`:

In [42]:
df1.reset_index(drop=True)

,PIB
0,1999
1,916
2,1374
3,359
4,1497
5,1665
6,1657
7,1286


Também se pode descartar todo um nível de indices (linhas ou colunas) através do método `droplevel()`:

In [43]:
df1.droplevel(0)

,PIB
Países,
China,1999
India,916
Japão,1374
Portugal,359
Alemanha,1497
Reino Unido,1665
U.S.A.,1657
Canadá,1286


In [44]:
saude

paciente      Roberto               Ana              José            
medida            BPM Temperatura   BPM Temperatura   BPM Temperatura
ano  consulta                                                        
2023 2           58.0        38.5  46.0        37.5  85.0        38.5
     1           67.0        37.5  67.0        37.7  58.0        37.8
2022 2           61.0        37.6  46.0        37.4  79.0        37.1
     1           43.0        37.6  61.0        38.1  70.0        37.7

In [45]:
saude.droplevel(1, axis=1)

paciente       Roberto  Roberto   Ana   Ana  José  José
ano  consulta                                          
2023 2            58.0     38.5  46.0  37.5  85.0  38.5
     1            67.0     37.5  67.0  37.7  58.0  37.8
2022 2            61.0     37.6  46.0  37.4  79.0  37.1
     1            43.0     37.6  61.0  38.1  70.0  37.7

Em "sentido contrário", ou seja, transformar colunas de dados em índices pode ser feito através do método `set_index`, indicando a(s) coluna(s) a transformar em índices. Por defeito o índice anterior é removido, a não ser que se utilize o parâmetro `append=True`, e nesse caso a(s) coluna(s) indicadas serão adicionadas como um novo nível de um índice hierárquico.

In [46]:
df2

,Continentes,Países,PIB
0,Asia,China,1999
1,Asia,India,916
2,Asia,Japão,1374
3,Europa,Portugal,359
4,Europa,Alemanha,1497
5,Europa,Reino Unido,1665
6,América,U.S.A.,1657
7,América,Canadá,1286


In [47]:
df2.set_index(['Continentes', 'Países'])

PIB
Continentes Países           
Asia        China        1999
            India         916
            Japão        1374
Europa      Portugal      359
            Alemanha     1497
            Reino Unido  1665
América     U.S.A.       1657
            Canadá       1286

In [48]:
df3 = df2.set_index('Continentes')
df3

,Países,PIB
Continentes,,
Asia,China,1999
Asia,India,916
Asia,Japão,1374
Europa,Portugal,359
Europa,Alemanha,1497
Europa,Reino Unido,1665
América,U.S.A.,1657
América,Canadá,1286


In [49]:
df3 = df3.set_index('Países', append=True)
df3

PIB
Continentes Países           
Asia        China        1999
            India         916
            Japão        1374
Europa      Portugal      359
            Alemanha     1497
            Reino Unido  1665
América     U.S.A.       1657
            Canadá       1286

"Transpor" um `DataFrame` pode ser feito com o método `transpose`, ou com o atributo especial `T`. Mas atenção que transpor um `DataFrame` pode fazer com que a informação sobre o tipo da dados se perca, e portanto transpor a transposta  pode fazer com que não se volte "ao ponto de inicial", pois a informação sobre os tipos de dados pode ser perdida.

Considere o seguinte `DataFrame`, com os tipos de dados definidos:

In [50]:
orig = (
    df2.loc[[0,3,6]]
    .astype({'Continentes': 'string', 'Países': 'string', 'PIB':'Int64'})
    .reset_index(drop=True)
)
orig

,Continentes,Países,PIB
0,Asia,China,1999
1,Europa,Portugal,359
2,América,U.S.A.,1657


Podemos confirmar os tipos de dados em cada uma das colunas do `DataFrame`:

In [51]:
orig.dtypes

Continentes    string
Países         string
PIB             Int64
dtype: object

Vamos então transpor utilizando o atributo especial `T`:

In [52]:
transposta = orig.T
transposta

,0,1,2
Continentes,Asia,Europa,América
Países,China,Portugal,U.S.A.
PIB,1999,359,1657


E voltar a transpos a transposta (voltando ao formato inicial), deste vez utilizando o método `transpose`:

In [53]:
nova = transposta.transpose()
nova

,Continentes,Países,PIB
0,Asia,China,1999
1,Europa,Portugal,359
2,América,U.S.A.,1657


Podemos confirmar que os dados são exatamente os mesmos:

In [54]:
orig == nova

,Continentes,Países,PIB
0,True,True,True
1,True,True,True
2,True,True,True


Mas os tipos de dados foram "perdidos":

In [55]:
nova.dtypes

Continentes    object
Países         object
PIB            object
dtype: object

In [56]:
orig.dtypes == nova.dtypes

Continentes    False
Países         False
PIB            False
dtype: bool

## Combinar e juntar _datasets_

Frequentemente é necessário combinar diferentes _datasets_ existentes, de forma a criar um único `DataFrame` para poder analisar os dados.

O pandas tem várias funções para combinar os seus objectos, entre outras vamos abordar as seguintes:

* `pandas.merge` - Combina linhas de `DataFrame` com base em chaves. É similar ao `JOIN` das bases de dados relacionais
* `pandas.concat` - Junta ou "encaixa" objectos do pandas com base num determinado eixo

### Combinar `DataFrames` de forma semelhante às Bases de Dados

Conside os seguintes `DataFrame`:

In [57]:
c5 = ['beta', 'beta', 'alfa', 'gama', 'alfa']
c3 = ['alfa', 'beta', 'delta']

In [58]:
df1 = pd.DataFrame({'chave': c5, 'dados1': range(5)}, index=list('abcde'))
df1

,chave,dados1
a,beta,0
b,beta,1
c,alfa,2
d,gama,3
e,alfa,4


In [59]:
df2 = pd.DataFrame({'chave': c3, 'dados2': range(3)}, index=list('xyz'))
df2

,chave,dados2
x,alfa,0
y,beta,1
z,delta,2


A função `merge` permite "juntar" dois `DataFrame`, efetuando o que no mundo das bases de dados se chama um _join_. Por defeito, caso não se indique quais as colunas a usar como chaves, o pandas vai utilizar todas as colunas que existam em comum nos dois objectos, mas é sempre boa pratica indicar qual/quais as colunas a utilizar. Atenção que quando se faz `merge` nas colunas, **os índices serão ignorados** e descartados. Caso seja importante manter esses índices deve-se fazer o `reset_index` para incluir esses índices como colunas de dados.

In [60]:
pd.merge(df1, df2, on='chave')

,chave,dados1,dados2
0,beta,0,1
1,beta,1,1
2,alfa,2,0
3,alfa,4,0


Caso não existam colunas com o mesmo nome nos dois `DataFrame`, pode-se indicar o nome da coluna a utilizar em cada um, com os parâmetros `left_on` e `right_on`. Por exemplo:

In [61]:
df3 = pd.DataFrame({'chave_esq': c5, 'dados': range(5)})
df3

,chave_esq,dados
0,beta,0
1,beta,1
2,alfa,2
3,gama,3
4,alfa,4


In [62]:
df4 = pd.DataFrame({'chave_dir': c3, 'dados': range(3)})
df4

,chave_dir,dados
0,alfa,0
1,beta,1
2,delta,2


In [63]:
pd.merge(df3, df4, left_on='chave_esq', right_on='chave_dir')

,chave_esq,dados_x,chave_dir,dados_y
0,beta,0,beta,1
1,beta,1,beta,1
2,alfa,2,alfa,0
3,alfa,4,alfa,0


Acima também se pode observar que, como a coluna dos dados tinha o mesmo nome nos dois `DataFrame`, o pandas utilizou os sufixos `_x` e `_y`, para cada uma das colunas. Caso se queira indicar um sufixo específico pode-se utilizar o parâmetro `suffixes`, passando uma lista/tuplo com duas strings, uma para cada `DataFrame`.

In [64]:
pd.merge(df3, df4, left_on='chave_esq', right_on='chave_dir', suffixes=('_esq', '_dir'))

,chave_esq,dados_esq,chave_dir,dados_dir
0,beta,0,beta,1
1,beta,1,beta,1
2,alfa,2,alfa,0
3,alfa,4,alfa,0


Por defeito, o `merge` faz o `inner` _join_, ou seja, as chaves no resultado são a intersecção das chaves nos dois `DataFrame`, i.e., apenas aparecem as chaves comuns aos dois. Também se pode fazer o `left`, o `right` e o `outer` _join_, utilizando o parâmetro `how`.

In [65]:
pd.merge(df1, df2, on='chave', how='inner')

,chave,dados1,dados2
0,beta,0,1
1,beta,1,1
2,alfa,2,0
3,alfa,4,0


In [66]:
pd.merge(df1, df2, on='chave', how='outer')

,chave,dados1,dados2
0,alfa,2.0,0.0
1,alfa,4.0,0.0
2,beta,0.0,1.0
3,beta,1.0,1.0
4,delta,NaN,2.0
5,gama,3.0,NaN


In [67]:
pd.merge(df3, df4, left_on='chave_esq', right_on='chave_dir', how='outer')

,chave_esq,dados_x,chave_dir,dados_y
0,alfa,2.0,alfa,0.0
1,alfa,4.0,alfa,0.0
2,beta,0.0,beta,1.0
3,beta,1.0,beta,1.0
4,NaN,NaN,delta,2.0
5,gama,3.0,NaN,NaN


No `outer` _join_, como se pode ver acima, as chaves que só existam em um dos `DataFrame` irão aparecer como valores em falta (_missing values_) nas colunas do outro `DataFrame`. De forma equivalente, no `left` _join_ irão aparecer todas as chaves do `DataFrame` da esquerda, e caso não existam no da direita, irão aparecer como `NA` nas colunas da direta, o mesmo acontecendo com o `right` _join_, mas de forma inversa.

In [68]:
pd.merge(df1, df2, on='chave', how='left')

,chave,dados1,dados2
0,beta,0,1.0
1,beta,1,1.0
2,alfa,2,0.0
3,gama,3,NaN
4,alfa,4,0.0


In [69]:
pd.merge(df1, df2, on='chave', how='right')

,chave,dados1,dados2
0,alfa,2.0,0
1,alfa,4.0,0
2,beta,0.0,1
3,beta,1.0,1
4,delta,NaN,2


Acima viu-se como se pode combinar diferentes `DataFrames`, com base em colunas de dados existentes, mas alguns casos a chave para usar no `merge` pode estar no índice. Nesses casos deve-se utilizar os parâmetros `left_index=True` e/ou `right_index=True`, para indicar que se deve utilizar os índices do `DataFrame` da esquerda e/ou da direita, respetivamente.

Por exemplo, considere os seguintes `DataFrame`:

In [70]:
clientes_dict = {
    'ncc': [987654321, 564738291, 123456789, 918273645],
    'nome': ['Barbara', 'Silva', 'Maria', 'Manel'],
    'codpostal': [8000, 8100, 8000, pd.NA]
}

clientes = pd.DataFrame(clientes_dict).set_index('ncc')
clientes

,nome,codpostal
ncc,,
987654321,Barbara,8000
564738291,Silva,8100
123456789,Maria,8000
918273645,Manel,<NA>


In [71]:
localidades_dict = {
    'codpostal': [8000, 8100, 8200, 8500],
    'localidade': ['Faro', 'Loulé', 'Albufeira', 'Portimão']
}
localidades = pd.DataFrame(localidades_dict).set_index('codpostal')
localidades

,localidade
codpostal,
8000,Faro
8100,Loulé
8200,Albufeira
8500,Portimão


Para poder fazer o `merge` destes `DataFrame` é necessário utilizar os parâmetros `left_on='codpostal'` e `right_index=True`, pois quer-se fazer o _join_ da coluna `codpostal` do primeiro `DataFrame` com o índice do segundo `DataFrame`:

In [72]:
pd.merge(clientes, localidades, left_on='codpostal', right_index=True, how='outer')

,nome,codpostal,localidade
987654321.0,Barbara,8000,Faro
123456789.0,Maria,8000,Faro
564738291.0,Silva,8100,Loulé
NaN,NaN,8200,Albufeira
NaN,NaN,8500,Portimão
918273645.0,Manel,<NA>,NaN


No exemplo acima pode-se ver que neste caso, os índices do `DataFrame` da esquerda foram mantidos, ao contrário do que acontecia acima. Isto acontece porque, neste caso, o índice da direita é único, e então o `merge` (com o `how='inner'` por defeito) consegue perservar os índices da esquerda que correspondem às linhas do resultado.

No caso de índices hierárquicos, o `merge` pode ser um pouco mais complicado, mas basicamente é equivalente a fazer um `merge` indicando um lista de chaves.

Considere os seguintes `DataFrame`:

In [73]:
df1 = pd.DataFrame({'chave1': ['Faro', 'Faro', 'Faro', 'Loulé', 'Loulé'],
                    'chave2': [2000, 2001, 2002, 2001, 2002],
                    'dados': pd.Series(range(5), dtype='Int64')})

In [74]:
index_df2 = pd.MultiIndex.from_arrays([
    ['Loulé', 'Loulé', 'Faro', 'Faro', 'Faro', 'Faro'],
    [2001, 2000, 2000, 2000, 2001, 2002]
])

df2 = pd.DataFrame({'evento1': pd.Series([0, 2, 4, 6, 8, 10],
                                         dtype='Int64', index=index_df2),
                    'evento2': pd.Series([1, 3, 5, 7, 9, 11],
                                         dtype='Int64', index=index_df2)})

In [75]:
df1

,chave1,chave2,dados
0,Faro,2000,0
1,Faro,2001,1
2,Faro,2002,2
3,Loulé,2001,3
4,Loulé,2002,4


In [76]:
df2

evento1  evento2
Loulé 2001        0        1
      2000        2        3
Faro  2000        4        5
      2000        6        7
      2001        8        9
      2002       10       11

Como no segundo `DataFrame` o índice tem dois níveis, para fazer o `merge` utilizando o índice, é necessário indicar duas colunas do primeiro `DataFrame` com o qual se ver fazer o `merge`:

In [77]:
pd.merge(df1, df2, left_on=['chave1', 'chave2'], right_index=True)

,chave1,chave2,dados,evento1,evento2
0,Faro,2000,0,4,5
0,Faro,2000,0,6,7
1,Faro,2001,1,8,9
3,Loulé,2001,3,0,1
2,Faro,2002,2,10,11


In [78]:
pd.merge(df1, df2, left_on=['chave1', 'chave2'], right_index=True, how='outer')

,chave1,chave2,dados,evento1,evento2
0,Faro,2000,0,4,5
0,Faro,2000,0,6,7
1,Faro,2001,1,8,9
2,Faro,2002,2,10,11
4,Loulé,2000,<NA>,2,3
3,Loulé,2001,3,0,1
4,Loulé,2002,4,<NA>,<NA>


## Concatenar ao longo de um eixo

Outra forma de "juntar" objectos é efetuando a concatenação dos dados ao longo de um determinado eixo, i.e., "juntar" linhas  ou colunas ao objecto. Isso pode ser feito com a função `concat`.

Em determinados casos, esta forma de concatenar pode levantar algumas dúvidas sobre o que fazer:

* O resultado da concatenação precisa de ser identificável, ou seja, é preciso saber quais os elementos que vieram de onde?
* O eixo que vamos usar para concatenar contém dados que é preciso preservar?

Considere as seguintes `Series`, cujos índices não se sobrepõem:

In [79]:
s1 = pd.Series([0, 1], index=["a", "b"], dtype="Int64")
s1

a    0
b    1
dtype: Int64

In [80]:
s2 = pd.Series([2, 3, 4], index=["c", "d", "e"], dtype="Int64", name='s2')
s2

c    2
d    3
e    4
Name: s2, dtype: Int64

In [81]:
s3 = pd.Series([5, 6], index=["f", "g"], dtype="Int64", name='s3')
s3

f    5
g    6
Name: s3, dtype: Int64

Por defeito a função `concat` recebe uma lista de objectos a concatenar e faz essa concatenação utilizando as linhas. Caso se pretenda fazer com as colunas deve-se utilizar o parâmetro `axis=1`:

In [82]:
pd.concat([s1, s2, s3])

a    0
b    1
c    2
d    3
e    4
f    5
g    6
dtype: Int64

In [83]:
pd.concat([s1, s2, s3], axis=1)

,0,s2,s3
a,0,<NA>,<NA>
b,1,<NA>,<NA>
c,<NA>,2,<NA>
d,<NA>,3,<NA>
e,<NA>,4,<NA>
f,<NA>,<NA>,5
g,<NA>,<NA>,6


Caso seja preciso indentificar o resultado, pode-ser criar um índice hirárquico no eixo utilizado para a concatenação, utilizando o parâmetro `keys`:

In [84]:
pd.concat([s1, s2, s3], keys=['s1', 's2', 's3'])

s1  a    0
    b    1
s2  c    2
    d    3
    e    4
s3  f    5
    g    6
dtype: Int64

No caso de se concatenar nas colunas, as `keys` serão os nomes das colunas:

In [85]:
pd.concat([s1, s2, s3], axis=1, keys=['aaa', 'bbb', 'ccc'])

,aaa,bbb,ccc
a,0,<NA>,<NA>
b,1,<NA>,<NA>
c,<NA>,2,<NA>
d,<NA>,3,<NA>
e,<NA>,4,<NA>
f,<NA>,<NA>,5
g,<NA>,<NA>,6


A mesma lógica é aplicável nos `DataFrame`:

In [86]:
df1 = pd.DataFrame([[0,1],[2,3],[4,5]], index=["a", "b", "c"],
                   columns=["um", "dois"])
df1

,um,dois
a,0,1
b,2,3
c,4,5


In [87]:
df2 = pd.DataFrame([[5,6],[7,8]], index=["a", "c"],
                   columns=["dois", "tres"])
df2

,dois,tres
a,5,6
c,7,8


In [88]:
df3 = pd.concat([df1, df2], axis="columns", keys=["nivel1", "nivel2"])
df3

nivel1      nivel2     
      um dois   dois tres
a      0    1    5.0  6.0
b      2    3    NaN  NaN
c      4    5    7.0  8.0

Se em vez de uma lista, passarmos um dicionário de objectos, as chaves do dicionário serão utilizadas como se fosse o parâmetro `keys`:

In [89]:
pd.concat({'nivel1': df1, 'nivel2': df2}, axis=1)

nivel1      nivel2     
      um dois   dois tres
a      0    1    5.0  6.0
b      2    3    NaN  NaN
c      4    5    7.0  8.0

Quando os índices das linhas não têm dados importantes, pode-se passar o parâmetro `ignore_index=True`, que faz com os índices do eixo da concatenação sejam ignorados, e seja criado um novo índice númerico

In [90]:
pd.concat([df1, df2], axis=0, ignore_index=True)

,um,dois,tres
0,0.0,1,NaN
1,2.0,3,NaN
2,4.0,5,NaN
3,NaN,5,6.0
4,NaN,7,8.0


In [91]:
pd.concat([df1, df2], axis=1, ignore_index=True)

,0,1,2,3
a,0,1,5.0,6.0
b,2,3,NaN,NaN
c,4,5,7.0,8.0


## Pivotar e Reestruturar

Pode-se restruturar a forma de um objecto do pandas através dos métodos `stack` ou `unstack`. O primeiro "roda" um dos índice das colunas (ou vários, caso seja passado uma lista no parâmetro `level`) de um `DataFrame` para índice da linha, e o segundo faz a operação inversa (linhas para colunas).

![](imagens/reshaping_stack.png)

![](imagens/reshaping_unstack.png)

Considere, por exemplo, o seguinte `DataFrame`:

In [92]:
df1

,um,dois
a,0,1
b,2,3
c,4,5


O método `stack` transforma os índices das colunas (`um` e `dois`) para um sub-nível dos índices das linhas (criando um índice hierárquico) e transformando, neste caso, o `DataFrame` numa `Series`:

In [93]:
s1 = df1.stack()
s1

a  um      0
   dois    1
b  um      2
   dois    3
c  um      4
   dois    5
dtype: int64

O `unstack` transforma, por defeito, o índice das linhas do nível interior do objecto no índice das colunas:

In [94]:
s1.unstack()

,um,dois
a,0,1
b,2,3
c,4,5


Pode-se utilizar o parâmetro `level` para indicar o nível do índice a ser transformado:

In [95]:
s1.unstack(level=0)

,a,b,c
um,0,2,4
dois,1,3,5


No caso de existir um índice hierárquico, podemos indicar qual o(s) índice(s) que queremos usar, sendo utilizado o último nível (nível mais interior) por defeito:

In [96]:
df3

nivel1      nivel2     
      um dois   dois tres
a      0    1    5.0  6.0
b      2    3    NaN  NaN
c      4    5    7.0  8.0

In [97]:
df3.stack()

nivel1  nivel2
a um       0.0     NaN
  dois     1.0     5.0
  tres     NaN     6.0
b um       2.0     NaN
  dois     3.0     NaN
  tres     NaN     NaN
c um       4.0     NaN
  dois     5.0     7.0
  tres     NaN     8.0

Caso `level` seja uma lista, serão utilizados os índices nessas posições:

In [98]:
s3 = df3.stack(level=[0,1], future_stack=True)
s3

a  nivel1  um      0.0
           dois    1.0
   nivel2  dois    5.0
           tres    6.0
b  nivel1  um      2.0
           dois    3.0
   nivel2  dois    NaN
           tres    NaN
c  nivel1  um      4.0
           dois    5.0
   nivel2  dois    7.0
           tres    8.0
dtype: float64

Atenção que o conceito "linhas" e "colunas" deixa de fazer sentido quando apenas existe "uma dimensão". Por exemplo, fazer o unstack (passa de linhas para colunas) quando apenas existe um nível de índices nas linhas transforma o `DataFrame` numa `Series`:

In [99]:
df3

nivel1      nivel2     
      um dois   dois tres
a      0    1    5.0  6.0
b      2    3    NaN  NaN
c      4    5    7.0  8.0

In [100]:
df3.unstack()

nivel1  um    a    0.0
              b    2.0
              c    4.0
        dois  a    1.0
              b    3.0
              c    5.0
nivel2  dois  a    5.0
              b    NaN
              c    7.0
        tres  a    6.0
              b    NaN
              c    8.0
dtype: float64

### Pivotar do formato "longo" para o formato "largo"

A forma tradicional de guardar dados, especialmente mas não só series temporais, numa base de dados ou num ficheiro CSV é chamado de formato "longo", que consiste em cada observação ser armazenada numa linha, em vez se utilizarem múltiplos valores em cada linha.

Considere os dados no seguinte ficheiro:

In [101]:
!type exemplos\indicadores.csv

regiÃ£o,indicador,2000
Norte,PIB_real,20000.0
Norte,inflaÃ§Ã£o,1.500
Norte,desemprego,7.200
Centro,PIB_real,15000.0
Centro,inflaÃ§Ã£o,1.800
Centro,desemprego,6.300
Lisboa,PIB_real,30000.0
Lisboa,inflaÃ§Ã£o,1.700
Lisboa,desemprego,5.100
Algarve,PIB_real,10000.0
Algarve,inflaÃ§Ã£o,2.200
Algarve,desemprego,8.000


In [102]:
df = pd.read_csv(r'exemplos\indicadores.csv')
df = df.sort_values(['região', 'indicador']).reset_index(drop=True)
df

,região,indicador,2000
0,Algarve,PIB_real,10000.0
1,Algarve,desemprego,8.0
2,Algarve,inflação,2.2
3,Centro,PIB_real,15000.0
4,Centro,desemprego,6.3
5,Centro,inflação,1.8
6,Lisboa,PIB_real,30000.0
7,Lisboa,desemprego,5.1
8,Lisboa,inflação,1.7
9,Norte,PIB_real,20000.0


Este formado denomina-se formato longo pois são "longas" colunas de valores, onde cada linha é uma observação.

Este formato é por vezes utilizado nas bases de dados pois permite adicionar valores diferentes à coluna `indicador`, conforme novos dados são adicionados à tabela.

No entanto, em certos casos pode ser problemático trabalhar com os dados neste formato, e seria preferível ter uma coluna para cada valor existente em `indicador`.

Isso pode ser feito com o método `pivot`, indicando qual as colunas que devem ser utilizadas para o(s) índice(s) das linhas e para o(s) índice(s) das colunas.

In [103]:
df.pivot(index='região', columns='indicador')

2000                    
indicador PIB_real desemprego inflação
região                                
Algarve    10000.0        8.0      2.2
Centro     15000.0        6.3      1.8
Lisboa     30000.0        5.1      1.7
Norte      20000.0        7.2      1.5

Como não se indicou qual/quais a(s) colunas  utilizar como valores da tabela, são utilizadas todas as restantes colunas, criando um índice hierárquico. Caso se indique apenas uma coluna no parâmetro `values` apenas os valores dessa coluna serão utilizados/apresentados:

In [104]:
df.pivot(index='região', columns='indicador', values='2000')

indicador,PIB_real,desemprego,inflação
região,,,
Algarve,10000.0,8.0,2.2
Centro,15000.0,6.3,1.8
Lisboa,30000.0,5.1,1.7
Norte,20000.0,7.2,1.5


Se não se quiser que, neste caso, a coluna `região` sejam o índice no `DataFrame` final, mas "volte" a ser uma coluna de dados, pode-se utilizar o `reset_index`:

In [105]:
df.pivot(index='região', columns='indicador', values='2000').reset_index()

indicador,região,PIB_real,desemprego,inflação
0,Algarve,10000.0,8.0,2.2
1,Centro,15000.0,6.3,1.8
2,Lisboa,30000.0,5.1,1.7
3,Norte,20000.0,7.2,1.5


Caso exista mais do que uma coluna de valores, irá ser criado um índice hierárquico com as colunas indicadas:

In [106]:
df['2020'] = range(1000,1012)
df

,região,indicador,2000,2020
0,Algarve,PIB_real,10000.0,1000
1,Algarve,desemprego,8.0,1001
2,Algarve,inflação,2.2,1002
3,Centro,PIB_real,15000.0,1003
4,Centro,desemprego,6.3,1004
5,Centro,inflação,1.8,1005
6,Lisboa,PIB_real,30000.0,1006
7,Lisboa,desemprego,5.1,1007
8,Lisboa,inflação,1.7,1008
9,Norte,PIB_real,20000.0,1009


In [107]:
df2 = df.pivot(index='região', columns='indicador', values=['2000', '2020'])
df2

2000                         2020                    
indicador PIB_real desemprego inflação PIB_real desemprego inflação
região                                                             
Algarve    10000.0        8.0      2.2   1000.0     1001.0   1002.0
Centro     15000.0        6.3      1.8   1003.0     1004.0   1005.0
Lisboa     30000.0        5.1      1.7   1006.0     1007.0   1008.0
Norte      20000.0        7.2      1.5   1009.0     1010.0   1011.0

Eventualmente mais alguns transformações podem tornam os dados ainda mais fáceis de trabalhar/perceber:

In [108]:
(
    df.pivot(index='região', columns='indicador', values=['2000', '2020'])
    .swaplevel(axis=1)
    .sort_index(axis=1)
    .reset_index()
)

indicador   região PIB_real         desemprego         inflação        
                       2000    2020       2000    2020     2000    2020
0          Algarve  10000.0  1000.0        8.0  1001.0      2.2  1002.0
1           Centro  15000.0  1003.0        6.3  1004.0      1.8  1005.0
2           Lisboa  30000.0  1006.0        5.1  1007.0      1.7  1008.0
3            Norte  20000.0  1009.0        7.2  1010.0      1.5  1011.0

### Pivotar do formato "largo" para o formato "longo"

A operação inversa do `pivot` é o `melt`, que "derrete" várias colunas numa única, "misturando" os valores e devolvendo um `DataFrame` que será mais "longo" e menos "largo".

![](imagens/reshaping_melt.png)

In [109]:
df2

2000                         2020                    
indicador PIB_real desemprego inflação PIB_real desemprego inflação
região                                                             
Algarve    10000.0        8.0      2.2   1000.0     1001.0   1002.0
Centro     15000.0        6.3      1.8   1003.0     1004.0   1005.0
Lisboa     30000.0        5.1      1.7   1006.0     1007.0   1008.0
Norte      20000.0        7.2      1.5   1009.0     1010.0   1011.0

Neste caso, o índice das linhas faz parte dos dados, logo é preciso indicar o parâmetro `ignore_index=False`, caso contrário o `melt` irá ignorar os índices.

In [110]:
df3 = df2.melt(ignore_index=False).reset_index().sort_values(['região', 'indicador'])
df3

,região,NaN,indicador,value
0,Algarve,2000,PIB_real,10000.0
12,Algarve,2020,PIB_real,1000.0
4,Algarve,2000,desemprego,8.0
16,Algarve,2020,desemprego,1001.0
8,Algarve,2000,inflação,2.2
20,Algarve,2020,inflação,1002.0
1,Centro,2000,PIB_real,15000.0
13,Centro,2020,PIB_real,1003.0
5,Centro,2000,desemprego,6.3
17,Centro,2020,desemprego,1004.0


Vamos indicar os nomes das colunas corretas:

In [111]:
df3.columns = ['região', 'ano', 'indicador', 'valor']
df3

,região,ano,indicador,valor
0,Algarve,2000,PIB_real,10000.0
12,Algarve,2020,PIB_real,1000.0
4,Algarve,2000,desemprego,8.0
16,Algarve,2020,desemprego,1001.0
8,Algarve,2000,inflação,2.2
20,Algarve,2020,inflação,1002.0
1,Centro,2000,PIB_real,15000.0
13,Centro,2020,PIB_real,1003.0
5,Centro,2000,desemprego,6.3
17,Centro,2020,desemprego,1004.0


E obtemos os nossos dados em "formato longo", uma observação em cada linha.

Caso se queira "voltar" à `DataFrame` inicial (com duas colunas), teremos que voltar a pivotar:

In [112]:
df3 = df3.pivot(index=['região', 'indicador'], columns='ano', values='valor').reset_index()
df3.columns.name = None
df3

,região,indicador,2000,2020
0,Algarve,PIB_real,10000.0,1000.0
1,Algarve,desemprego,8.0,1001.0
2,Algarve,inflação,2.2,1002.0
3,Centro,PIB_real,15000.0,1003.0
4,Centro,desemprego,6.3,1004.0
5,Centro,inflação,1.8,1005.0
6,Lisboa,PIB_real,30000.0,1006.0
7,Lisboa,desemprego,5.1,1007.0
8,Lisboa,inflação,1.7,1008.0
9,Norte,PIB_real,20000.0,1009.0


Podemos confirmar que os `DataFrame` são iguais:

In [113]:
(df == df3).all()

região       True
indicador    True
2000         True
2020         True
dtype: bool

Utilizando os parâmetros `id_vars`, `var_name` e `value_name`, pode-se indicar quais as colunas do `DataFrame` que são as variáveis identificadoras (basicamente, as que não contêm valores), qual o nome da coluna da variável (irá ter os valores dos nomes das colunas, e por defeito será `variable`) e qual o nome da coluna de dados (por defeito será `value`), respetivamente. Utilizando o `melt` na `DataFrame` inicial:

In [114]:
df.head()

,região,indicador,2000,2020
0,Algarve,PIB_real,10000.0,1000
1,Algarve,desemprego,8.0,1001
2,Algarve,inflação,2.2,1002
3,Centro,PIB_real,15000.0,1003
4,Centro,desemprego,6.3,1004


In [115]:
df.melt(id_vars=['região', 'indicador'], var_name='ano', value_name='valor').head(10)

,região,indicador,ano,valor
0,Algarve,PIB_real,2000,10000.0
1,Algarve,desemprego,2000,8.0
2,Algarve,inflação,2000,2.2
3,Centro,PIB_real,2000,15000.0
4,Centro,desemprego,2000,6.3
5,Centro,inflação,2000,1.8
6,Lisboa,PIB_real,2000,30000.0
7,Lisboa,desemprego,2000,5.1
8,Lisboa,inflação,2000,1.7
9,Norte,PIB_real,2000,20000.0
